In [16]:
# ! pip install 2to3
# !pip install DI-engine
# !apt install swig -y
# !pip install gym[box2d]==0.25.1
# # Current stable release of DI-engine
# !pip install git+https://github.com/opendilab/DI-engine.git@main#egg=DI-engine
# !pip install gym[box2d]
# !pip install pyecharts
# !pip install ufal.pybox2d
# !pip install swig
# !pip install gymnasium[box2d]
# !pip install imageio-ffmpeg

In [4]:
# !pip list

In [5]:
import os
os.chdir('..')

In [6]:
import gym # Load the gym library, which is used to standardize the reinforcement learning environment
import torch # Load the PyTorch library for loading the Tensor model and defining the computing network
from easydict import EasyDict # Load EasyDict for instantiating configuration files
from ding.config import compile_config # Load configuration related components in DI-engine config module
from ding.envs import DingEnvWrapper # Load environment related components in DI-engine env module
from ding.policy import DQNPolicy, single_env_forward_wrapper # Load policy-related components in DI-engine policy module
from ding.model import DQN # Load model related components in DI-engine model module
from dizoo.box2d.lunarlander.config.lunarlander_dqn_config import main_config, create_config # Load DI-zoo lunarlander environment and DQN algorithm related configurations


def main(main_config: EasyDict, create_config: EasyDict, ckpt_path: str):
    main_config.exp_name = 'lunarlander_dqn_deploy' # Set the name of the experiment to be run in this deployment, which is the name of the project folder to be created
    cfg = compile_config(main_config, create_cfg=create_config, auto=True) # Compile and generate all configurations
    env = DingEnvWrapper(gym.make(cfg.env.env_id), EasyDict(env_wrapper='default')) # Add the DI-engine environment decorator upon the gym's environment instance
    env.enable_save_replay(replay_path='./lunarlander_dqn_deploy/video') # Enable the video recording of the environment and set the video saving folder
    model = DQN(**cfg.policy.model) # Import model configuration, instantiate DQN model
    state_dict = torch.load(ckpt_path, map_location='cpu') # Load model parameters from file
    model.load_state_dict(state_dict['model']) # Load model parameters into the model
    policy = DQNPolicy(cfg.policy, model=model).eval_mode # Import policy configuration, import model, instantiate DQN policy, and turn to evaluation mode
    forward_fn = single_env_forward_wrapper(policy.forward) # Use the strategy decorator of the simple environment to decorate the decision method of the DQN strategy
    obs = env.reset() # Reset the initialization environment to get the initial observations
    returns = 0. # Initialize total reward
    while True: # Let the agent's strategy and environment interact cyclically until the end
        action = forward_fn(obs) # According to the observed state, make a decision and generate action
        obs, rew, done, info = env.step(action) # Execute actions, interact with the environment, get the next observation state, the reward of this interaction, the signal of whether to end, and other information
        returns += rew # Cumulative reward return
        if done:
            break
    print(f'Deploy is finished, final epsiode return is: {returns}')

/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/treevalue/tree/integration/torch.py:23: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  register_for_torch(TreeValue)
/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/treevalue/tree/integration/torch.py:24: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  register_for_torch(FastTreeValue)


[07-08 12:27:52] WARNING  If you want to use numba to speed up segment tree, please install numba first                                              ]8;id=461396;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/default_helper.py\default_helper.py]8;;\:]8;id=628576;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/default_helper.py#450\450]8;;\

[07-08 12:27:52] WARNING  Please install pyecharts first, you can install it by running 'pip install pyecharts'                                        ]8;id=452035;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/memory_helper.py\memory_helper.py]8;;\:]8;id=943847;file:///home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/ding/utils/memory_helper.py#12\12]8;;\

/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
os.listdir()

['README.md', 'notebooks', 'models', '.gitignore', '.git', 'requirements.txt']

In [17]:
main(main_config=main_config, create_config=create_config, ckpt_path='models/final.pth.tar')

/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/gym/wrappers/record_video.py:78: UserWarning: WARN: Overwriting existing videos at /home/azm0269@auburn.edu/ducks/lunarlander_dqn_deploy/video folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/home/azm0269@auburn.edu/.pyenv/versions/3.10.18/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Deploy is finished, final epsiode return is: [219.12772]
